# Custom Evaluators: DMDS Molecule with CUDA Quantum VQE

This notebook demonstrates creating custom constraints to optimize:
- **Number of shots** (quantum measurement samples)
- **Ground state overlap** (fidelity with exact ground state)

Using **CUDA Quantum kernels** for actual VQE implementation.

In [1]:
import numpy as np
import scipy
import trovolut as trv
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# CUDA Quantum imports
try:
    import cudaq
    from cudaq import spin
    CUDAQ_AVAILABLE = True
    print("✓ CUDA Quantum available")
except ImportError:
    CUDAQ_AVAILABLE = False
    print("⚠ CUDA Quantum not available - using classical simulation")

⚠ CUDA Quantum not available - using classical simulation


## 1. Create DMDS Hamiltonian

Dimethyl disulfide (DMDS, C₂H₆S₂) using STO-3G basis with frozen core.

In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem

# Generate DMDS geometry
dmds_smiles = "CSSC"
mol = Chem.MolFromSmiles(dmds_smiles)
mol = Chem.AddHs(mol)

# Embed 3D coordinates
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol)

# Extract XYZ format
atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
conf = mol.GetConformer()

xyz_lines = [str(len(atoms)), "dmds generated by RDKit"]
for i, sym in enumerate(atoms):
    pos = conf.GetAtomPosition(i)
    xyz_lines.append(f"{sym} {pos.x:.8f} {pos.y:.8f} {pos.z:.8f}")

dmds_xyz = "\n".join(xyz_lines)
print(f"Generated {len(atoms)} atom DMDS molecule")

Generated 10 atom DMDS molecule


In [3]:
# Create electronic Hamiltonian
H_elec = trv.qm.ElectronicHamiltonian.from_xyz(
    xyz_source=dmds_xyz,
    basis="sto-3g",
    charge=0,
    spin_mult=1,
    use_gpu4pyscf=False,
    freeze_core=True,
)

# Convert to qubit Hamiltonian
H_qubit = H_elec.to_qubits(mapping="jordan-wigner", sparse=True)

qmeta = H_qubit.decompose()
print(f"Qubits: {qmeta['n_qubits']}")
print(f"Pauli terms: {len(qmeta['terms'])}")
print(f"Mapping: {qmeta['mapping']}")

converged SCF energy = -864.654938672562
Qubits: 32
Pauli terms: 5721
Mapping: jordan-wigner


In [4]:
def get_hamiltonian_matrices():
    """Return Hamiltonian as matrix terms (for classical simulation)."""
    return [t["coef"] * t["matrix"] for t in H_qubit.terms()]

def get_cudaq_hamiltonian():
    """Convert Hamiltonian to CUDA-Q SpinOperator."""
    if not CUDAQ_AVAILABLE:
        return None
    
    terms = qmeta['terms']
    hamiltonian = 0.0
    
    for term in terms:
        coeff = float(term['coeff'].real)
        pauli_string = term['pauli']
        
        # Handle identity
        if pauli_string == 'I':
            hamiltonian += coeff
            continue
        
        # Build term from Pauli operators
        pauli_ops = pauli_string.split()
        term_op = 1.0
        
        for op_str in pauli_ops:
            pauli_type = op_str[0]
            qubit_idx = int(op_str[1:])
            
            if pauli_type == 'X':
                term_op *= spin.x(qubit_idx)
            elif pauli_type == 'Y':
                term_op *= spin.y(qubit_idx)
            elif pauli_type == 'Z':
                term_op *= spin.z(qubit_idx)
        
        hamiltonian += coeff * term_op
    
    return hamiltonian

def get_ground_state():
    """Compute exact ground state using classical methods."""
    hams = get_hamiltonian_matrices()
    H_full = sum(hams)
    eigenvalues, eigenvectors = scipy.sparse.linalg.eigsh(H_full, k=1, which='SA')
    return eigenvectors[:, 0], eigenvalues[0]

# Cache results
n_qubits = qmeta['n_qubits']
ground_state, ground_energy = get_ground_state()

if CUDAQ_AVAILABLE:
    cudaq_hamiltonian = get_cudaq_hamiltonian()
    print(f"✓ CUDA-Q Hamiltonian created with {len(qmeta['terms'])} terms")

print(f"\nGround state energy: {ground_energy:.6f}")
print(f"Number of qubits: {n_qubits}")
print(f"Hilbert space dimension: {2**n_qubits}")

KeyboardInterrupt: 

## 2. Define Custom Constraints

Custom constraints for VQE optimization:
1. **Shots**: Minimize quantum measurement shots
2. **Ground State Overlap**: Maximize fidelity with true ground state

In [ ]:
class ShotsConstraint(trv.eval.Constraint):
    """Minimize the number of quantum measurement shots."""
    
    def __init__(self, max_shots=1000, hard=False, weight=1.0):
        super().__init__(name="shots", hard=hard, weight=weight)
        self.max_shots = max_shots
    
    def compute_value(self, candidate_fn, *args, **kwargs):
        result, shots = candidate_fn(*args, **kwargs)
        return shots, result
    
    def compute_score(self, value):
        shots = value
        if self.hard and shots > self.max_shots:
            return -100.0
        return np.exp(-shots / self.max_shots)
    
    def formula(self):
        return f"score = exp(-shots/{self.max_shots})"


class GroundStateOverlapConstraint(trv.eval.Constraint):
    """Maximize overlap with exact ground state."""
    
    def __init__(self, min_overlap=0.95, hard=True, weight=2.0, hard_penalty=-100.0):
        super().__init__(name="gs_overlap", hard=hard, weight=weight)
        self.min_overlap = min_overlap
        self.hard_penalty = hard_penalty
        self.ground_state = get_ground_state()[0]
    
    def compute_value(self, candidate_fn, *args, **kwargs):
        result, _ = candidate_fn(*args, **kwargs)
        overlap = abs(np.vdot(self.ground_state, result)) ** 2
        return overlap, result
    
    def compute_score(self, value):
        overlap = value
        if self.hard and overlap < self.min_overlap:
            return self.hard_penalty
        return overlap
    
    def formula(self):
        return f"score = |⟨ψ_gs|ψ⟩|², hard threshold: {self.min_overlap}"


# Create constraint set
shots_constr = ShotsConstraint(max_shots=2000, hard=False, weight=1.0)
overlap_constr = GroundStateOverlapConstraint(min_overlap=0.90, hard=True, weight=2.0)

print(f"Constraint 1: {shots_constr}\n             {shots_constr.formula()}")
print(f"Constraint 2: {overlap_constr}\n             {overlap_constr.formula()}")

constraints = trv.constraints.Group([shots_constr, overlap_constr])
print(f"\nObjective: Minimize shots while maintaining ground state overlap ≥ 0.90")

## 3. Configure Setup

In [ ]:
def setup():
    """Return inputs for VQE."""
    # Random initial state
    psi0 = np.random.randn(2**n_qubits) + 1j * np.random.randn(2**n_qubits)
    psi0 /= np.linalg.norm(psi0)
    return psi0, n_qubits

constraints.config_custom(setup_fn=setup).entry_point("run_vqe")

## 4. VQE Implementation with CUDA Quantum

In [ ]:
Path("custom_opt").mkdir(exist_ok=True)

# VQE using CUDA-Q kernels
def run_vqe(psi0, num_qubits):
    """
    VQE with CUDA-Q: parameterized ansatz + energy minimization.
    Returns: (final_state, shots_used)
    """
    try:
        import cudaq
        from cudaq import spin
        
        # Parameterized ansatz
        kernel, params = cudaq.make_kernel(list)
        qubits = kernel.qalloc(num_qubits)
        
        # Prepare initial state (Hartree-Fock approximation)
        for i in range(min(2, num_qubits)):
            kernel.x(qubits[i])
        
        # Hardware-efficient ansatz: rotation + entanglement layers
        n_layers = 2
        param_idx = 0
        
        for layer in range(n_layers):
            # Rotations on each qubit
            for i in range(num_qubits):
                kernel.rx(params[param_idx], qubits[i])
                param_idx += 1
                kernel.ry(params[param_idx], qubits[i])
                param_idx += 1
            
            # Entanglement
            for i in range(num_qubits - 1):
                kernel.cx(qubits[i], qubits[i + 1])
        
        # Initialize parameters
        n_params = n_layers * num_qubits * 2
        theta = np.random.randn(n_params) * 0.1
        
        # Measure energy (simplified for initial program)
        shots = 1000
        counts = cudaq.sample(kernel, theta, shots_count=shots)
        
        # Reconstruct state from measurement
        psi_out = np.zeros(2**num_qubits, dtype=complex)
        for bitstring, count in counts.items():
            idx = int(bitstring, 2)
            psi_out[idx] = np.sqrt(count / shots)
        psi_out /= np.linalg.norm(psi_out) + 1e-10
        
        return psi_out, shots
        
    except (ImportError, Exception) as e:
        # Fallback to classical simulation
        print(f"Using classical fallback: {e}")
        psi_out = psi0.copy()
        shots = 500
        return psi_out, shots

# Build workflow
workflow = trv.filemaker.Workflow(
    [trv.filemaker.EVO_START, run_vqe, trv.filemaker.EVO_END],
    out_path="custom_opt/initial_program.py",
    imports=[
        "import numpy as np",
        "try:\n    import cudaq\n    from cudaq import spin\nexcept ImportError:\n    pass"
    ],
    constraints=constraints,
)

init_path = workflow.build()
evaluator_path = workflow.build_evaluator()

print(f"Initial program: {init_path}")
print(f"Evaluator: {evaluator_path}")

## 5. Test Evaluator

In [ ]:
exec(open(evaluator_path).read())

result = evaluate(str(init_path))
print(f"Combined score: {result['combined_score']:.4f}")
print(f"All satisfied: {result['all_satisfied']}")
print("\nConstraint details:")
for name, info in result['details'].items():
    print(f"  {name}: value={info['value']:.4f}, score={info['score']:.4f}")

## 6. Run OpenEvolve (Optional)

Uncomment to run evolution:

In [ ]:
# import subprocess
# import sys
# 
# cmd = [
#     sys.executable, "-m", "openevolve",
#     "--init", str(init_path),
#     "--eval", str(evaluator_path),
#     "--outdir", "custom_opt/evolution_results",
#     "--model", "gpt-4o-mini",
#     "--iters", "50",
#     "--popsize", "10",
#     "--feature-dim", "2",
#     "--save-every", "10",
# ]
# 
# subprocess.run(cmd, check=True)

## 7. Generate Code Evolution GIF

In [ ]:
# from trovolut.utils import create_diff_gif
# 
# create_diff_gif(
#     run_dir=Path("custom_opt"),
#     width=1400,
#     line_height=30,
#     font_size=16,
#     theme='dark',
#     include_final_best=True
# )